# NepalDTM-G: all remaining provinces + national 1 m map tiles

Runs Koshi, Madhesh, Gandaki, Lumbini, Karnali and Sudurpashchim one after another (Bagmati is done), uploading each
finished 10 km tile to R2 bucket `nepaldtm-g-1m/<province>/tiles/`, then builds the national map archives
`nepaldtm-g-1m/nepal/NepalDTM_G_nepal_<k>.webp.pmtiles` that the website map uses.

**Setup:** Runtime → Change runtime type → **A100 GPU**, **High-RAM** on. Colab secrets (key icon, notebook access on):
`R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_ENDPOINT`.

**Resume:** after any disconnect, Runtime → Run all. Finished provinces, tiles already in R2 and uploaded map archives
are skipped; at most the tile in progress is lost. It prints `ALL DONE` at the end.

In [ ]:
import os
from google.colab import userdata
for k in ("R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_ENDPOINT"):
    os.environ[k] = userdata.get(k)
os.environ["R2_BUCKET"] = "nepaldtm-g-1m"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!pip -q install rasterio shapely pyproj boto3 affine pmtiles
!rm -rf /content/NepalDTM && git clone -q --depth 1 -b g https://github.com/achyut/NepalDTM /content/NepalDTM

Batch size: 64 on an 80 GB A100, 32 on a 40 GB A100 (picked automatically), 16 if it runs out of GPU memory.

In [ ]:
import subprocess
mem = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).split()[0])
BS = 64 if mem > 60000 else 32
print("batch size", BS)
!cd /content/NepalDTM/code && python -u g/g_colab_all.py --bs {BS}